In [11]:
from sqlalchemy import create_engine, text
import pandas as pd
from pathlib import Path
import mlflow
import time
import requests
from secrets_local import DB_URL
from src.config import MLFLOW_TRACKING_URI, PIPELINE_SCHEMA_VERSION
from src.app.db_querriees import db_connector
from src.models.train_model import verify_model

In [ ]:
con = db_connector(DB_URL)

from src.features.build_features import (
    drop_features,
    add_new_features,
    make_dummies,
)


def booler(x):
    if x in ["No", 0]:
        return False
    elif x in ["Yes", 1]:
        return True


def preproc(df):
    df = add_new_features(df)
    df = make_dummies(df)
    df = drop_features(df)
    return df

In [ ]:
con.update_drift_report(window_id=2, target_drift=False, concept_drift=True)

In [ ]:
MLFLOW_TRACKING_URI = "http://localhost:5000"


def load_model() -> dict | None:
    mlflow_availibility = False
    attempts = 0
    while not mlflow_availibility and attempts < 12:
        try:
            requests.get(MLFLOW_TRACKING_URI, timeout=2.0)
            mlflow_availibility = True
        except requests.RequestException:
            mlflow_availibility = False
        attempts += 1

    if mlflow_availibility:
        mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

        runs = mlflow.search_runs(
            search_all_experiments=True,
            filter_string=(
                "tags.production = 'True' AND "
                "tags.model_saved = 'True' AND "
                f"tags.pipeline_schema_version = '{PIPELINE_SCHEMA_VERSION}'"
            ),
            order_by=["metrics.roc_auc DESC"],
            max_results=1,
        )

        if len(runs) == 0:
            print("There are no suitable model on MLflow server")
            print("App will start in test mode, without model")
            return None
        else:
            run_id = runs.iloc[0]["run_id"]
            model_uri = f"runs:/{run_id}/model"
            model = mlflow.sklearn.load_model(model_uri)
            result = {"model": model, "run_id": run_id}
            return result
    else:
        print("MLflow server unavailible, cant load model")
        print(f"({attempts} attempts to connect)")
        print("App will start in test mode, without model")
        return None

In [34]:
win_data = con.get_window_rows(1471, 2940)

train_data = con.get_train_data("539506ca4ae941c7b539b1a3f3f6d2aa")

In [29]:
target_train = train_data["GT_Attrition"].copy().apply(booler)
target_win = win_data["GT_Attrition"].copy().apply(booler)


train_data = train_data.drop(
    columns=[
        "row_id",
        "model_train_run_id",
        "pipeline_shema_version",
        "GT_Attrition",
    ]
)
win_data = win_data.drop(columns=["row_id", "GT_Attrition"])

train_data = preproc(train_data)
win_data = preproc(win_data)

model = load_model()["model"]

In [30]:
mertics_train = verify_model(model, train_data, target_train, verbose=False)
metrics_win = verify_model(model, win_data, target_win, verbose=False)

e:\miniconda\envs\mlops\Lib\site-packages\sklearn\metrics\_ranking.py:469: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
e:\miniconda\envs\mlops\Lib\site-packages\sklearn\metrics\_classification.py:614: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
e:\miniconda\envs\mlops\Lib\site-packages\sklearn\metrics\_ranking.py:469: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
e:\miniconda\envs\mlops\Lib\site-packages\sklearn\metrics\_classification.py:614: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


In [33]:
target_win.describe()

count    1470.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
Name: GT_Attrition, dtype: float64

In [9]:
win_data

,row_id,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,...,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,GT_Attrition
0,1471,33,1,Travel_Rarely,1277,Research & Development,15,1,Medical,1,...,80,0,15,2,4,7,6,7,7,1
1,1472,36,0,Travel_Frequently,1099,Research & Development,2,2,Medical,1,...,80,1,5,3,3,10,2,6,0,0
2,1473,40,1,Travel_Rarely,448,Research & Development,16,3,Life Sciences,1,...,80,0,18,2,2,4,2,3,3,0
3,1474,29,0,Travel_Rarely,992,Research & Development,1,3,Technical Degree,1,...,80,0,7,1,2,6,2,1,5,1
4,1475,29,0,Travel_Frequently,723,Research & Development,20,3,Life Sciences,1,...,80,1,11,5,3,1,7,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1465,2936,36,1,Travel_Rarely,311,Research & Development,7,3,Life Sciences,1,...,80,0,15,4,3,4,3,1,3,0
1466,2937,40,0,Travel_Rarely,654,Research & Development,27,3,Medical,1,...,80,1,10,2,3,5,4,0,1,0
1467,2938,40,0,Travel_Rarely,1124,Sales,1,2,Medical,1,...,80,3,6,2,2,4,3,0,2,0
1468,2939,28,1,Travel_Rarely,760,Sales,2,4,Marketing,1,...,80,0,8,2,3,8,7,7,5,0


In [3]:
con.get_earliest_window_with_new_labels()

,window_id,w_start,w_stop,run_id,data_drift,target_drift,concept_drift,trained


In [8]:
rrr = {
    "window_id": 2,
    "w_start": 1471,
    "w_stop": 2940,
    "run_id": "539506ca4ae941c7b539b1a3f3f6d2aa",
    "data_drift": False,
    "target_drift": None,
    "concept_drift": None,
    "trained": False,
}
res = con.get_window_rows(rrr["w_start"], rrr["w_stop"])

In [9]:
res

,row_id,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,...,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,GT_Attrition


In [3]:
df = pd.read_csv(
    Path.cwd().parent / "data" / "raw" / "IBM_HR_attr.csv",
    index_col="Unnamed: 0",
)
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [4]:
indexes = {}

for key in ["train", "val", "test"]:
    indexes[key] = pd.read_csv(
        Path.cwd().parent / "data" / "processed" / f"dataset_{key}.csv",
        index_col="Unnamed: 0",
    ).index

In [5]:
# put train data in db
mlflow_availibility = True
try:
    requests.get("http://localhost:5000/", timeout=10.0)
except:
    Exception("MLFLOW unavailible")

if mlflow_availibility:
    mlflow.set_tracking_uri("http://localhost:5000/")
    runs = mlflow.search_runs(
        search_all_experiments=True,
        filter_string=(
            "tags.production = 'True' AND "
            "tags.model_saved = 'True' AND "
            f"tags.pipeline_schema_version = '{PIPELINE_SCHEMA_VERSION}'"
        ),
        order_by=["metrics.roc_auc DESC"],
        max_results=1,
    )

    if len(runs) == 0:
        print("There are no suitable model on MLflow server")
        print("App will start in test mode, without model")
        status = "alive, in testing mode"
    else:
        run_id = runs.iloc[0]["run_id"]
        status = "alive"

print("status:", status)
print("run_id:", run_id)

for part in ["train", "val", "test"]:
    for idx in range(len(indexes[part])):
        row = df.loc[idx].to_dict()
        if type(row["Attrition"]) == str:
            row["Attrition"] = 1 if row["Attrition"] == "Yes" else 0
        row["model_train_run_id"] = "None"

        row_id = con.insert_history(row)

        row_train_hist = {
            "model_train_run_id": run_id,
            "row_id": row_id,
            "pipeline_shema_version": PIPELINE_SCHEMA_VERSION,
            "data_part": part,
        }
        con.insert_train_history(row_train_hist)

        row_gt = {"row_id": row_id, "GT_Attrition": row["Attrition"]}

        con.insert_gt_labels(row_gt)
        time.sleep(1 / 200)

print("done")

status: alive
run_id: 539506ca4ae941c7b539b1a3f3f6d2aa
done


In [ ]:
engine = create_engine(DB_URL)


def history_insert(row):
    with engine.begin() as conn:
        result = conn.execute(
            text("""
                INSERT INTO history (
                    model_train_run_id,
                    Age,
                    Attrition,
                    BusinessTravel,
                    DailyRate,
                    Department,
                    DistanceFromHome,
                    Education,
                    EducationField,
                    EmployeeCount,
                    EmployeeNumber,
                    EnvironmentSatisfaction,
                    Gender,
                    HourlyRate,
                    JobInvolvement,
                    JobLevel,
                    JobRole,
                    JobSatisfaction,
                    MaritalStatus,
                    MonthlyIncome,
                    MonthlyRate,
                    NumCompaniesWorked,
                    Over18,
                    OverTime,
                    PercentSalaryHike,
                    PerformanceRating,
                    RelationshipSatisfaction,
                    StandardHours,
                    StockOptionLevel,
                    TotalWorkingYears,
                    TrainingTimesLastYear,
                    WorkLifeBalance,
                    YearsAtCompany,
                    YearsInCurrentRole,
                    YearsSinceLastPromotion,
                    YearsWithCurrManager
                )
                VALUES (
                    :model_train_run_id,
                    :Age,
                    :Attrition,
                    :BusinessTravel,
                    :DailyRate,
                    :Department,
                    :DistanceFromHome,
                    :Education,
                    :EducationField,
                    :EmployeeCount,
                    :EmployeeNumber,
                    :EnvironmentSatisfaction,
                    :Gender,
                    :HourlyRate,
                    :JobInvolvement,
                    :JobLevel,
                    :JobRole,
                    :JobSatisfaction,
                    :MaritalStatus,
                    :MonthlyIncome,
                    :MonthlyRate,
                    :NumCompaniesWorked,
                    :Over18,
                    :OverTime,
                    :PercentSalaryHike,
                    :PerformanceRating,
                    :RelationshipSatisfaction,
                    :StandardHours,
                    :StockOptionLevel,
                    :TotalWorkingYears,
                    :TrainingTimesLastYear,
                    :WorkLifeBalance,
                    :YearsAtCompany,
                    :YearsInCurrentRole,
                    :YearsSinceLastPromotion,
                    :YearsWithCurrManager
                )
                RETURNING history.row_id
            """),
            row,
        )
        row_id = result.scalar_one()
        return row_id

In [10]:
def train_history_insert(row):
    with engine.begin() as conn:
        conn.execute(
            text("""
                INSERT INTO train_history (
                    model_train_run_id,
                    row_id,
                    pipeline_shema_version,
                    data_part
                )
                VALUES (
                    :model_train_run_id,
                    :row_id,
                    :pipeline_shema_version,
                    :data_part
                )
            """),
            row,
        )

In [11]:
def gt_labels_insert(row):
    with engine.begin() as conn:
        conn.execute(
            text("""
                INSERT INTO gt_labels (
                    row_id,
                    GT_Attrition
                )
                VALUES (
                    :row_id,
                    :GT_Attrition
                )
            """),
            row,
        )

In [19]:
row2 = {
    "model_train_run_id": run_id,
    "row_id": 1,
    "pipeline_shema_version": 1,
    "data_part": "valid",
}

train_history_insert(row2)

In [24]:
row3 = {"row_id": 1, "GT_Attrition": 0}

gt_labels_insert(row3)

In [13]:
# put train data in db
mlflow_availibility = True
try:
    requests.get("http://localhost:5000/", timeout=10.0)
except:
    Exception("MLFLOW unavailible")

if mlflow_availibility:
    mlflow.set_tracking_uri("http://localhost:5000/")
    runs = mlflow.search_runs(
        search_all_experiments=True,
        filter_string=(
            "tags.production = 'True' AND "
            "tags.model_saved = 'True' AND "
            f"tags.pipeline_schema_version = '{PIPELINE_SCHEMA_VERSION}'"
        ),
        order_by=["metrics.roc_auc DESC"],
        max_results=1,
    )

    if len(runs) == 0:
        print("There are no suitable model on MLflow server")
        print("App will start in test mode, without model")
        status = "alive, in testing mode"
    else:
        run_id = runs.iloc[0]["run_id"]
        status = "alive"

print("status:", status)
print("run_id:", run_id)

for part in ["train", "val", "test"]:
    for idx in range(len(indexes[part])):
        row = df.loc[idx].to_dict()
        if type(row["Attrition"]) == str:
            row["Attrition"] = 1 if row["Attrition"] == "Yes" else 0
        row["model_train_run_id"] = run_id

        row_id = history_insert(row)

        row_train_hist = {
            "model_train_run_id": run_id,
            "row_id": row_id,
            "pipeline_shema_version": PIPELINE_SCHEMA_VERSION,
            "data_part": part,
        }
        train_history_insert(row_train_hist)

        row_gt = {"row_id": row_id, "GT_Attrition": row["Attrition"]}

        gt_labels_insert(row_gt)


print("done")

status: alive
run_id: 539506ca4ae941c7b539b1a3f3f6d2aa
done
